# Teach a tiny model to answer

Write about fifty short dialogues, train a LoRA adapter on **HuggingFaceTB/SmolLM2-135M (base)**, and compare the same prompts before and after. The supplied examples invent a “pocket workshop” helper: brief, concrete, and fond of “Pocket note:” and “Try:”. These are agent-written teaching examples, not student transcripts.

You need no API key and no prior training experience. You will edit JSON, run cells, and look closely at language. Fifty examples can teach a recognizable habit; they do not establish reliable instruction following, truthfulness, or safety.

1. Install and choose a device.
2. Load your examples and the unchanged base model.
3. Capture base continuations, then train only the adapter.
4. Compare, save, and optionally export to ONNX.

**Colab:** upload this notebook with File → Upload notebook, then Runtime → Change runtime type → T4 GPU (or another available GPU). Run cells in order. The data cell asks you to upload `sample.json` or your completed template. A GPU is optional locally. Downloads and installation take additional time; the training cell measures training separately.


In [ ]:
import os, subprocess, sys

# Pinned to the compatible stack tested with this notebook. Colab supplies torch.
# Local reproduction installs requirements.txt first and skips this cell's install.
if os.environ.get("NOTEBOOK_SKIP_INSTALL") != "1":
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "torch>=2.6", "transformers==4.57.6", "peft==0.21.2",
        "trl==1.16.0", "datasets==5.0.1", "accelerate==1.15.0",
    ])
print("Dependencies ready. If Colab asks for a restart, restart the session and rerun from here.")


## 1. Set the experiment

Start with the supplied data and 80 optimizer steps. Then change the data, restart the runtime, and run again from the base weights. `STEPS=20` is a smoke test, not evidence of a useful assistant. The environment overrides below let the local runner execute the same notebook without editing it. CPU and Apple MPS use float32; CUDA uses mixed precision. We avoid CUDA-only quantization packages.


In [ ]:
from pathlib import Path
from time import perf_counter
import importlib.metadata as metadata
import json, platform

ROOT = Path.cwd()
if (ROOT / "fine-tune" / "dialogues").exists():
    ROOT = ROOT / "fine-tune"
OUTPUT = ROOT / "outputs"
OUTPUT.mkdir(exist_ok=True)
os.environ.setdefault("HF_HOME", str(ROOT / ".cache" / "huggingface"))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from datasets import Dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
from IPython.display import HTML, display
from html import escape

MODEL_ID = "HuggingFaceTB/SmolLM2-135M"  # deliberately NOT the Instruct version
MODEL_REVISION = "93efa2f097d58c2a74874c7e644dbc9b0cee75a2"  # tested base snapshot
STEPS = int(os.environ.get("FINE_TUNE_STEPS", "80"))
DATA_FILE = Path(os.environ.get("FINE_TUNE_DATA", str(ROOT / "dialogues" / "sample.json")))
EXPORT_ONNX = os.environ.get("FINE_TUNE_EXPORT_ONNX", "0") == "1"  # optional after training
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
set_seed(42)
torch.set_num_threads(4)

def synchronize():
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    elif DEVICE == "mps":
        torch.mps.synchronize()

versions = {name: metadata.version(name) for name in ("torch", "transformers", "peft", "trl", "datasets", "accelerate")}
print(json.dumps({"device": DEVICE, "python": platform.python_version(), "versions": versions, "steps": STEPS}, indent=2))
session_started = perf_counter()


## 2. Load dialogues you can read and edit

`dialogues/template.json` is a JSON array of objects with `user` and `assistant` fields. Fill the blanks, duplicate the objects until you have about fifty, and save as `my-dialogues.json`. Change `DATA_FILE` above, or upload that file when prompted in Colab. Upload **one** JSON file. Blank templates, embedded turn labels, and overly long examples are rejected rather than silently used for training.

Choose a voice and repeat its habits across varied requests. Include uncertainty, disagreement, and endings as well as greetings. Keep a few new questions outside the training data so you can tell imitation from memorization. Do not put private conversations into a teaching dataset.


In [ ]:
if not DATA_FILE.exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f"No dialogue file at {DATA_FILE}. Set DATA_FILE to your JSON file.")
    uploaded = files.upload()
    candidates = [name for name in uploaded if name.endswith(".json")]
    if len(candidates) != 1:
        raise ValueError("Upload exactly one dialogue JSON file, then rerun this cell.")
    DATA_FILE = Path(candidates[0])

dialogues = json.loads(DATA_FILE.read_text(encoding="utf-8"))
if not isinstance(dialogues, list) or not dialogues:
    raise ValueError("Expected a nonempty JSON array of user/assistant objects.")
for i, row in enumerate(dialogues):
    if not isinstance(row, dict) or set(row) != {"user", "assistant"}:
        raise ValueError(f"Dialogue {i + 1}: use exactly the keys user and assistant.")
    for key, value in row.items():
        if not isinstance(value, str) or not value.strip():
            raise ValueError(f"Dialogue {i + 1}: fill in {key}.")
        if any(marker in value for marker in ("User:", "Assistant:", "<|endoftext|>")):
            raise ValueError(f"Dialogue {i + 1}: remove embedded turn/end labels.")
if not 40 <= len(dialogues) <= 60:
    print("For the full exercise, aim for 40–60 dialogues; continuing with", len(dialogues))
print(f"Loaded {len(dialogues)} dialogues from {DATA_FILE.name}")
print(json.dumps(dialogues[0], indent=2, ensure_ascii=False))


## 3. The format is part of the lesson

The base model is a text continuer. We give it literal `User:` and `Assistant:` labels, not an inherited instruct model's personality. Each training completion ends with the model's existing EOS token. We add no vocabulary items. TRL's `completion_only_loss=True` teaches the reply and its ending while masking the prompt. In this exercise each dialogue has one user turn and one response.

We also save a matching chat template for an eventual browser loader. The template and the plain-text helper below must produce the same string. Both comparisons use that string, with no extra system prompt and greedy decoding.


In [ ]:
load_started = perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, token=False)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{% if message['role'] == 'user' %}{{ 'User: ' + message['content'] + '\n' }}"
    "{% elif message['role'] == 'assistant' %}{{ 'Assistant: ' + message['content'] + eos_token }}"
    "{% else %}{{ raise_exception('Only user and assistant roles are supported') }}{% endif %}"
    "{% endfor %}{% if add_generation_prompt %}{{ 'Assistant:' }}{% endif %}"
)

def prompt_for(text):
    return f"User: {text}\nAssistant:"

assert tokenizer.apply_chat_template([{"role": "user", "content": "Hello."}], tokenize=False, add_generation_prompt=True) == prompt_for("Hello.")
training_rows = [{"prompt": prompt_for(row["user"]), "completion": " " + row["assistant"].strip() + tokenizer.eos_token} for row in dialogues]
lengths = [len(tokenizer(row["prompt"] + row["completion"], add_special_tokens=False)["input_ids"]) for row in training_rows]
if max(lengths) > 256:
    raise ValueError(f"Longest dialogue is {max(lengths)} tokens. Shorten it to <=256; nothing will be silently truncated.")
dataset = Dataset.from_list(training_rows)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, dtype=torch.float32, attn_implementation="eager", token=False,
).to(DEVICE)
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
synchronize()
load_seconds = perf_counter() - load_started
print(f"Base model loaded in {load_seconds:.1f}s; longest dialogue: {max(lengths)} tokens")
print("Exact evaluation prefix:", repr(prompt_for("Hello.")))


In [ ]:
# Five new prompts, plus one training prompt explicitly labeled as a memorization check.
PROMPTS = [
    "My sculpture keeps falling over. What should I try?",
    "How can I make the opening of my comic more surprising?",
    "I have three unfinished songs and an hour. Help me choose a next step.",
    "Can you promise that everyone will love my painting?",
    "What sort of helper are you?",
    dialogues[0]["user"],
]

@torch.inference_mode()
def continue_text(text):
    model.eval()
    inputs = tokenizer(prompt_for(text), return_tensors="pt", add_special_tokens=False).to(DEVICE)
    ids = model.generate(
        **inputs, max_new_tokens=64, do_sample=False, use_cache=True,
        eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id,
    )
    # Decode only the continuation, without concealing a model-written next user turn.
    return tokenizer.decode(ids[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)

base_outputs = [continue_text(prompt) for prompt in PROMPTS]
print("Base continuations captured before adding LoRA.")


## 4. Train a small adapter

The base weights stay frozen. LoRA adds low-rank matrices to the linear layers; the printed parameter count tells you how much is being updated. Eighty steps with batch size four revisit these fifty examples several times. This deliberately small exercise can overfit: a falling training loss alone does not mean a capable assistant has appeared.


In [ ]:
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.0,
    target_modules="all-linear", task_type="CAUSAL_LM", bias="none",
)
args = SFTConfig(
    output_dir=str(OUTPUT / "training"), max_steps=STEPS,
    per_device_train_batch_size=4, gradient_accumulation_steps=1,
    learning_rate=1e-3, lr_scheduler_type="linear", warmup_steps=min(5, STEPS // 4),
    max_length=256, packing=False, completion_only_loss=True,
    gradient_checkpointing=False, optim="adamw_torch",
    fp16=(DEVICE == "cuda"), bf16=False,
    logging_steps=10, save_strategy="no", report_to="none",
    dataloader_pin_memory=False, seed=42,
)
model.config.use_cache = False
trainer = SFTTrainer(
    model=model, args=args, train_dataset=dataset,
    processing_class=tokenizer, peft_config=lora,
)
model = trainer.model
model.print_trainable_parameters()
# Check that the trainer actually masks some prompt tokens and learns some reply tokens.
batch = trainer.data_collator([trainer.train_dataset[0]])
assert (batch["labels"] == -100).any() and (batch["labels"] != -100).any()
synchronize()
train_started = perf_counter()
result = trainer.train()
synchronize()
train_seconds = perf_counter() - train_started
print(f"Training: {train_seconds:.1f}s ({STEPS} optimizer steps, {DEVICE})")


## 5. Compare side by side

Read the first five rows as new questions. The last row is from the training set and can show memorization. Does the model answer the question, copy a phrase, repeat itself, or write both sides? “Pocket note:” is a visible style cue, not a score for intelligence. Greedy output is useful for controlled comparison; try sampling later as a separate experiment.


In [ ]:
model.config.use_cache = True
tuned_outputs = [continue_text(prompt) for prompt in PROMPTS]
comparison = [{"prompt": p, "in_training_set": p in {d["user"] for d in dialogues}, "base": b, "fine_tuned": t} for p, b, t in zip(PROMPTS, base_outputs, tuned_outputs)]
rows_html = "".join(
    "<tr>" + "".join(f"<td style='vertical-align:top;border:1px solid #888;padding:10px;white-space:pre-wrap'>{escape(str(value))}</td>" for value in (row["prompt"] + (" [training example]" if row["in_training_set"] else " [new]"), row["base"], row["fine_tuned"])) + "</tr>"
    for row in comparison
)
display(HTML("<div style='overflow:auto'><table style='width:100%;table-layout:fixed'><thead><tr><th>Prompt</th><th>Base</th><th>With LoRA</th></tr></thead><tbody>" + rows_html + "</tbody></table></div>"))
metrics = {
    "model": MODEL_ID, "model_revision": getattr(model.config, "_commit_hash", None),
    "device": DEVICE, "steps": STEPS, "dialogues": len(dialogues), "batch_size": 4,
    "load_seconds": load_seconds, "training_seconds": train_seconds,
    "session_seconds_through_comparison": perf_counter() - session_started,
    "versions": versions, "train_metrics": result.metrics,
}
(OUTPUT / "comparison.json").write_text(json.dumps(comparison, indent=2, ensure_ascii=False))
(OUTPUT / "metrics.json").write_text(json.dumps(metrics, indent=2))
print("Saved outputs/comparison.json and outputs/metrics.json")


## 6. Save the adapter and merge a copy for inference

The adapter is small and needs the original base model to run. The merged model includes the base weights and is much larger. Keep these outputs out of git. Merging removes the ability to disable the adapter on that object; we already captured the base comparison above. Export uses float32 on CPU for portability.


In [ ]:
ADAPTER = OUTPUT / "adapter"
MERGED = OUTPUT / "merged"
model.save_pretrained(ADAPTER)
tokenizer.save_pretrained(ADAPTER)
merged = model.to("cpu").merge_and_unload(safe_merge=True)
merged.config.use_cache = True
merged.save_pretrained(MERGED, safe_serialization=True)
tokenizer.save_pretrained(MERGED)
print("Saved adapter:", ADAPTER)
print("Saved merged model:", MERGED)


## 7. Optional: export an ONNX folder for Transformers.js

Leave `EXPORT_ONNX=False` for the first run. ONNX is a separate deployment step, not required for the before/after experiment. The README explains the tested export and browser compatibility check. This cell installs the optional exporter, exports with a KV cache, and moves the graph into Transformers.js's `onnx/model.onnx` layout. Do not upload a LoRA adapter directly to the browser engine. Uploading/publishing weights is your choice; nothing here publishes them.


In [ ]:
if EXPORT_ONNX:
    if os.environ.get("NOTEBOOK_SKIP_INSTALL") != "1":
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "optimum-onnx[onnxruntime]==0.1.0", "optimum==2.1.0"])
    from optimum.exporters.onnx import main_export
    import shutil
    export_started = perf_counter()
    BROWSER = OUTPUT / "browser-model"
    main_export(
        str(MERGED), output=BROWSER, task="text-generation-with-past",
        opset=18, device="cpu", dtype="fp32", do_validation=True,
    )
    (BROWSER / "onnx").mkdir(exist_ok=True)
    for path in BROWSER.glob("*.onnx*"):
        shutil.move(str(path), str(BROWSER / "onnx" / path.name))
    tokenizer.save_pretrained(BROWSER)
    # Transformers.js 4.3.1 reads the template from JSON, not the separate Jinja file.
    config_path = BROWSER / "tokenizer_config.json"
    browser_tokenizer_config = json.loads(config_path.read_text())
    browser_tokenizer_config["chat_template"] = tokenizer.chat_template
    config_path.write_text(json.dumps(browser_tokenizer_config, indent=2))
    from onnxruntime.quantization import QuantType, quantize_dynamic
    quantize_dynamic(
        str(BROWSER / "onnx" / "model.onnx"),
        str(BROWSER / "onnx" / "model_quantized.onnx"),
        weight_type=QuantType.QUInt8, op_types_to_quantize=["MatMul"],
    )
    print(f"Export completed in {perf_counter() - export_started:.1f}s")
    print("Browser folder:", BROWSER)
else:
    print("Skipped ONNX export. The adapter, merged model, and comparisons are already saved.")


## Try a second version

Change ten answers so the helper asks a question instead of giving a “Try:” instruction. Predict which prompts will change. Restart the runtime and rerun with the revised file from the unchanged base model. Fill in this observation before comparing:

> I changed ____. I expect ____ because ____. On a new prompt I observed ____. A copied phrase I noticed was ____.

**Common snags:** upload a completed template rather than blank fields; keep each example under 256 tokens; if CUDA runs out of memory lower batch size to 1; if you rerun the training cell, it continues from the current adapter, so restart for a fair new experiment. If there is no Colab GPU, select a GPU runtime or expect a slower CPU run. Download `outputs/adapter`, `outputs/comparison.json`, and `outputs/metrics.json` before Colab ends; its temporary filesystem is not permanent storage.

## API references checked for this notebook

- [Transformers model loading and saving](https://huggingface.co/docs/transformers/main_classes/model)
- [TRL SFTTrainer: prompt/completion data, completion-only loss, PEFT](https://huggingface.co/docs/trl/sft_trainer)
- [PEFT LoRA configuration](https://huggingface.co/docs/peft/package_reference/lora) and [merging adapters](https://huggingface.co/docs/peft/developer_guides/checkpoint)
- [Optimum ONNX export with past key values](https://huggingface.co/docs/optimum-onnx/onnx/usage_guides/export_a_model)
- [Transformers.js custom models](https://huggingface.co/docs/transformers.js/custom_usage)

The exact locally tested versions and timing are in `README.md`. Colab GPU time is an estimate until someone runs it there.
